# Création des Datasets

## 1. Imports

In [4]:
import os
import re
import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from scipy.stats import skew, kurtosis
import neurokit2 as nk
import importlib.metadata
import tsfel
import itertools

warnings.filterwarnings("ignore")


In [7]:
print(importlib.metadata.version("tsfel"))
nk.__version__

0.2.0


'0.2.13'

## 2. Config Générale

In [23]:
# ============================================================
# CHEMINS
# ============================================================

DOSSIER_PRETRAITES = Path(r"\\fss.ulaval.ca\partages\FSS\Départements\Psychologie"
    r"\Laboratoires de recherche\Attention\Neurostimulation\Data\Données Prétraitées")

CHEMIN_TEMPS = Path("Feuilles_de_Temps_TDCS_timestamps_ms.xlsx")

# utile seulement pour ECG/RSP post si Source_post == oxysoft
CHEMIN_TEMPS_BIO_POST = Path("Feuille_de_temps_only_SensorHub.xlsx")

CHEMIN_GROUPES = Path(
    r"\\fss.ulaval.ca\partages\FSS\Départements\Psychologie"
    r"\Laboratoires de recherche\Attention\Neurostimulation\Data\jeux de donnée.xlsx"
)

DOSSIER_DATASETS = Path("DATASETS_cover&interpolation")
DOSSIER_DATASETS.mkdir(parents=True, exist_ok=True)

# ============================================================
# FICHIERS PRÉTRAITÉS
# ============================================================

FICH = {
    "fNIRS": "BriteLiteFrontalTDDRFiltered_v3.csv",
    "ECG":   "BioEcgPreprocessed_v3.csv",
    "RSP":   "BioBreathingPreprocessed_v3.csv",
}

FS = {
    "fNIRS": 25,
    "ECG": 250,
    "RSP": 18,
}

# ============================================================
# PARAMÈTRES DE FENÊTRAGE À TESTER
# ============================================================

CONFIGS_FENETRAGE = [
    {
        "nom": "short_step10_nirs20_bio60",
        "step_ms": 10_000,
        "win_ms": {"fNIRS": 20_000, "ECG": 60_000, "RSP": 60_000},
    },
    {
        "nom": "medium_step15_nirs30_bio120",
        "step_ms": 15_000,
        "win_ms": {"fNIRS": 30_000, "ECG": 120_000, "RSP": 120_000},
    },
    {
        "nom": "long_step30_nirs60_bio180",
        "step_ms": 30_000,
        "win_ms": {"fNIRS": 60_000, "ECG": 180_000, "RSP": 180_000},
    },
]

DUREE_AVANT_MS = 5 * 60 * 1000
EDGE_MODE = "outside"
RULES_QUALITE = {
    "fNIRS": {"min_cover": 0.85, "max_gap_s": 2.0},
    "ECG":   {"min_cover": 0.80, "max_gap_s": 3.0},
    "RSP":   {"min_cover": 0.80, "max_gap_s": 5.0},
}

# ============================================================
# ÉVÉNEMENTS / LABELS
# ============================================================

KSS_ANCHORS = {
    "KSS1": "Pré - Fin P1",
    "KSS2": "Pré - Fin O1",
    "KSS3": "Pré - Fin O2",
    "KSS6": "Post - Fin P1",
}
#"KSS4": "Pré - Fin A1",
#"KSS5": "Pré - Fin A2",

MAP_KSS_VALUE = {
    "KSS1": "PRE_PVT_FATIGUE_KSS",
    "KSS2": "B1_MATB_FATIGUE_KSS",
    "KSS3": "B2_MATB_FATIGUE_KSS",
    "KSS6": "POST_PVT_FATIGUE_KSS",
}

#"KSS4": "B1_ANT_FATIGUE_KSS",
#"KSS5": "B2_ANT_FATIGUE_KSS",

SIGNALS_FNIRS = ["O2Hb", "HHb", "tHb", "HbDiff"]

HEMISPHERE = {
    "Rx1Tx1": "G", "Rx1Tx3": "G", "Rx2Tx1": "G", "Rx2Tx3": "G",
    "Rx3Tx4": "D", "Rx3Tx5": "D", "Rx4Tx4": "D", "Rx4Tx5": "D",
}

HRV_VOULUES = [
    # Time-domain
    "HRV_MeanNN", "HRV_SDNN", "HRV_RMSSD", "HRV_pNN50", "HRV_pNN20",
    "HRV_MedianNN", "HRV_MadNN", "HRV_CVNN",

    # Géométriques
    "HRV_HTI", "HRV_TINN",

    # Frequency-domain
    "HRV_VLF", "HRV_LF", "HRV_HF", "HRV_VHF",
    "HRV_LFHF", "HRV_LFn", "HRV_HFn",

    # Non-linéaires
    "HRV_SD1", "HRV_SD2", "HRV_SD1SD2",
    "HRV_DFA_alpha1", "HRV_DFA_alpha2",
    "HRV_SampEn",
]


INCLURE_RSA = True


## 3. Config TSFEL

In [24]:
FEATURES_TSFEL = [
    "Mean",
    "Median",
    "Standard deviation",
    "Variance",
    "Root mean square",
    "Slope",
    "Skewness",
    "Kurtosis",
    "Mean absolute deviation",
    "Max",
    "Min",
    "Peak to peak distance",
    "Area under the curve",
    "Absolute energy",
    "Entropy",
    "Zero crossing rate",
    "Fundamental frequency",
    "Maximum frequency",
    "Power bandwidth",
    "Spectral distance",
    "Median frequency",
    "Spectral entropy",
]

CFG_TSFEL = tsfel.get_features_by_domain(["statistical", "temporal", "spectral"])

for domaine in CFG_TSFEL:
    for feat in CFG_TSFEL[domaine]:
        CFG_TSFEL[domaine][feat]["use"] = "no"

wanted = {f.lower() for f in FEATURES_TSFEL}
trouvees = []

for domaine in CFG_TSFEL:
    for feat in CFG_TSFEL[domaine]:
        if feat.lower() in wanted:
            CFG_TSFEL[domaine][feat]["use"] = "yes"
            trouvees.append(feat)

print(f"Features TSFEL activées : {len(trouvees)}")


Features TSFEL activées : 22


## 4. Chargement feuilles temps et groupes

In [25]:
def extraire_num_participant(x):
    if pd.isna(x):
        return pd.NA
    m = re.search(r"\d+", str(x))
    return int(m.group()) if m else pd.NA


def charger_feuille_temps(path):
    df = pd.read_excel(path)
    df.columns = df.columns.str.strip()

    df["Participant"] = df["Participant"].apply(extraire_num_participant)
    df = df.dropna(subset=["Participant"]).copy()
    df["Participant"] = df["Participant"].astype(int)

    for c in ["Source_pre", "Source_post"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.lower().str.strip()

    for c in df.columns:
        if c not in ["Participant", "Date", "Source_pre", "Source_post", "Notes"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


df_temps = charger_feuille_temps(CHEMIN_TEMPS)

if CHEMIN_TEMPS_BIO_POST.exists():
    df_temps_bio_post = charger_feuille_temps(CHEMIN_TEMPS_BIO_POST)
else:
    df_temps_bio_post = pd.DataFrame()
    print("⚠️ Feuille BIO POST absente.")


In [26]:
# Groupes + labels KSS
df_groupes = pd.read_excel(CHEMIN_GROUPES)
df_groupes.columns = df_groupes.columns.str.strip()

df_groupes["Participant"] = (
    df_groupes["id"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .pipe(pd.to_numeric, errors="coerce")
)

df_groupes = df_groupes.dropna(subset=["Participant"]).copy()
df_groupes["Participant"] = df_groupes["Participant"].astype(int)

map_groupe = df_groupes.set_index("Participant")["group"]

kss_cols_exist = [c for c in MAP_KSS_VALUE.values() if c in df_groupes.columns]

kss_long = (
    df_groupes.set_index("Participant")[kss_cols_exist]
    .rename(columns={v: k for k, v in MAP_KSS_VALUE.items()})
    .stack()
    .rename("KSS_label")
    .reset_index()
    .rename(columns={"level_1": "KSS"})
)


## 5. Chargement des fichiers prétraités par participant

In [27]:
def dossier_participant(num):
    return DOSSIER_PRETRAITES / f"P{int(num):02d}"


def lire_csv_pretraite(path):
    if not path.exists():
        return None

    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = df.columns.str.strip()

    if "Time_ms" not in df.columns:
        return None

    df["Time_ms"] = pd.to_numeric(df["Time_ms"], errors="coerce")
    df = df.dropna(subset=["Time_ms"]).copy()

    # Normalisation colonne Bloc / bloc / Block
    bloc_cols = [c for c in df.columns if c.lower() in ["bloc", "block"]]
    if bloc_cols:
        col_bloc = bloc_cols[0]
        df["Bloc_norm"] = (
            df[col_bloc]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    return df


def charger_donnees_pretraitees():
    dico = {}

    participants = sorted(df_temps["Participant"].dropna().astype(int).unique())

    for num in tqdm(participants):
        dossier = dossier_participant(num)

        dico[num] = {
            "fNIRS": lire_csv_pretraite(dossier / FICH["fNIRS"]),
            "ECG":   lire_csv_pretraite(dossier / FICH["ECG"]),
            "RSP":   lire_csv_pretraite(dossier / FICH["RSP"]),
        }

    return dico


In [7]:
dico_data = charger_donnees_pretraitees()

100%|██████████| 60/60 [27:49<00:00, 27.82s/it]


In [28]:
for num, data in dico_data.items():
    df_n = data.get("fNIRS")

    if df_n is None:
        continue

    if "Bloc_norm" not in df_n.columns:
        continue

    print(f"\nP{num:02d}")
    print(df_n.groupby("Bloc_norm")["Time_ms"].agg(["min", "max", "count"]))



P01
                min        max  count
Bloc_norm                            
post            0.0  1356560.0  33915
pre        225760.0  4155920.0  98255

P02
                    min           max   count
Bloc_norm                                    
post       0.000000e+00  1.378280e+06   34458
pre        1.742930e+12  1.742934e+12  109563

P03
                    min           max   count
Bloc_norm                                    
post       0.000000e+00  1.339240e+06   33482
pre        1.743095e+12  1.743099e+12  100186

P04
                    min           max   count
Bloc_norm                                    
post       0.000000e+00  1.398280e+06   34958
pre        1.743109e+12  1.743113e+12  110470

P05
                     min            max  count
Bloc_norm                                     
post       1743172639182  1743173972327  33330
pre        1743167087325  1743171030828  98583

P06
                     min            max   count
Bloc_norm                     

In [29]:
def modalites_disponibles(num):
    d = dico_data.get(int(num), {})
    return [m for m in ["fNIRS", "ECG", "RSP"] if d.get(m) is not None]


df_dispo = pd.DataFrame([
    {
        "Participant": num,
        "fNIRS": dico_data[num]["fNIRS"] is not None,
        "ECG": dico_data[num]["ECG"] is not None,
        "RSP": dico_data[num]["RSP"] is not None,
    }
    for num in dico_data
])

display(df_dispo.head())
print(df_dispo[["fNIRS", "ECG", "RSP"]].sum())


,Participant,fNIRS,ECG,RSP
0,1,True,False,False
1,2,True,True,True
2,3,True,True,True
3,4,True,True,True
4,5,True,True,True


fNIRS    60
ECG      52
RSP      53
dtype: int64


In [30]:
def bloc_attendu_kss(kss_name):
    col = KSS_ANCHORS.get(kss_name, "").lower()

    if "post" in col:
        return "post"
    elif "pré" in col or "pre" in col:
        return "pre"
    else:
        return None


## 6. Gestion des temps fNIRS vs BIO

In [31]:
df_temps_idx = df_temps.set_index("Participant")

if not df_temps_bio_post.empty:
    df_temps_bio_post_idx = df_temps_bio_post.set_index("Participant")
else:
    df_temps_bio_post_idx = pd.DataFrame()


def get_row_temps(num, modalite, kss_name):
    """
    fNIRS : toujours feuille principale.
    ECG/RSP post + Source_post oxysoft : feuille bio post si disponible.
    """
    num = int(num)

    if num not in df_temps_idx.index:
        return None

    row_main = df_temps_idx.loc[num]

    if modalite in ["ECG", "RSP"] and kss_name.startswith("KSS6"):
        source_post = str(row_main.get("Source_post", "")).lower().strip()

        if (
            source_post == "oxysoft"
            and not df_temps_bio_post_idx.empty
            and num in df_temps_bio_post_idx.index
        ):
            return df_temps_bio_post_idx.loc[num]

    return row_main


def get_anchor_time(num, modalite, kss_name):
    col = KSS_ANCHORS[kss_name]
    row = get_row_temps(num, modalite, kss_name)

    if row is None or col not in row.index:
        return None

    v = row[col]
    return float(v) if pd.notna(v) else None


## 7. Moteur de fenêtrage

In [32]:
def centers_relatifs(duree_ms, step_ms, wmax_ms, edge="outside"):
    """
    Centres relatifs à l'événement cible.
    Ex : -300s -> 0s.
    """
    t1, t2 = -duree_ms, 0

    if edge == "inside":
        lo = t1 + wmax_ms / 2
        hi = t2 - wmax_ms / 2
    else:
        lo = t1 + step_ms / 2
        hi = t2

    if hi < lo:
        return np.array([])

    return np.arange(lo, hi + 1e-9, step_ms)


def slice_window(df, center_abs, win_ms, bloc=None):
    if df is None or "Time_ms" not in df.columns:
        return None

    d = df

    # Si une colonne Bloc_norm existe, on filtre dessus
    if bloc is not None and "Bloc_norm" in d.columns:
        d = d[d["Bloc_norm"] == bloc]

    lo = center_abs - win_ms / 2
    hi = center_abs + win_ms / 2

    return d[(d["Time_ms"] >= lo) & (d["Time_ms"] <= hi)].copy()


## 8. Extracteurs features

## fNIRS

In [33]:
def detecter_bases_fnirs(df):
    if df is None:
        return []

    bases = []

    for c in df.columns:
        cu = c.upper()
        if cu.endswith("O2HB"):
            bases.append(c[:-4])

    return sorted(set(bases))


def feats_fnirs_custom(y, fs=25):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]

    if len(y) < 2:
        return {
            "Custom_SlopeNaive": np.nan,
            "Custom_MaximumVariation": np.nan,
            "Custom_MinimumVariation": np.nan,
        }

    out = {}
    out["Custom_SlopeNaive"] = y[-1] - y[0]

    w = int(fs)

    if len(y) < w + 1:
        out["Custom_MaximumVariation"] = np.nan
        out["Custom_MinimumVariation"] = np.nan
        return out

    forward_mean = np.array([
        np.mean(y[i:i + w])
        for i in range(0, len(y) - w + 1)
    ])

    backward_mean = np.array([
        np.mean(y[i - w + 1:i + 1])
        for i in range(w - 1, len(y))
    ])

    out["Custom_MaximumVariation"] = np.max(np.diff(forward_mean))
    out["Custom_MinimumVariation"] = np.min(np.diff(backward_mean))

    return out


def feats_fnirs(sub, bases):
    out = {}

    for base in bases:
        col_o2 = f"{base}O2HB"
        col_hh = f"{base}HHB"

        if col_o2 not in sub.columns or col_hh not in sub.columns:
            continue

        o2 = sub[col_o2].to_numpy(float)
        hh = sub[col_hh].to_numpy(float)

        sigs = {
            "O2Hb": o2,
            "HHb": hh,
            "tHb": o2 + hh,
            "HbDiff": o2 - hh,
        }

        for sig_name in SIGNALS_FNIRS:
            y = sigs[sig_name]
            y = pd.Series(y).interpolate(limit_direction="both").to_numpy()

            try:
                df_f = tsfel.time_series_features_extractor(
                    CFG_TSFEL,
                    y,
                    fs=FS["fNIRS"],
                    verbose=0
                )

                for col in df_f.columns:
                    out[f"fNIRS_{base}_{sig_name}_{col}"] = df_f[col].iloc[0]

            except Exception:
                pass

            for col, val in feats_fnirs_custom(y, fs=FS["fNIRS"]).items():
                out[f"fNIRS_{base}_{sig_name}_{col}"] = val

    return out


## ECG

In [34]:
def feats_ecg(sub):
    out = {}

    if "ECG_clean" not in sub.columns:
        return out, None

    sig = sub["ECG_clean"].to_numpy(float)

    if len(sig) < FS["ECG"] * 20:
        return out, None

    _, info = nk.ecg_peaks(
        sig,
        sampling_rate=FS["ECG"],
        correct_artifacts=True
    )

    rpeaks = info.get("ECG_R_Peaks", [])

    if len(rpeaks) < 8:
        return out, None

    rate = nk.ecg_rate(
        rpeaks,
        sampling_rate=FS["ECG"],
        desired_length=len(sig)
    )

    out["ECG_HR_Mean"] = np.nanmean(rate)
    out["ECG_HR_Std"] = np.nanstd(rate)

    parts = []

    try:
        parts.append(nk.hrv_time(rpeaks, sampling_rate=FS["ECG"]))
    except Exception:
        pass

    try:
        parts.append(nk.hrv_frequency(rpeaks, sampling_rate=FS["ECG"]))
    except Exception:
        pass

    try:
        parts.append(nk.hrv_nonlinear(rpeaks, sampling_rate=FS["ECG"]))
    except Exception:
        pass

    if parts:
        hrv = pd.concat(parts, axis=1)
    
        for k in HRV_VOULUES:
            if k in hrv.columns:
                out[k] = hrv[k].iloc[0]
    
        # Alias demandé : HRV_DFA_alpha = HRV_DFA_alpha1
        if "HRV_DFA_alpha1" in hrv.columns:
            out["HRV_DFA_alpha"] = hrv["HRV_DFA_alpha1"].iloc[0]
    
    return out, rpeaks



## RSP

In [35]:
def feats_rsp(sub):
    out = {}

    if "RSP_clean" not in sub.columns:
        return out, None

    sig = sub["RSP_clean"].to_numpy(float)

    if len(sig) < FS["RSP"] * 30:
        return out, None

    rsp, _ = nk.rsp_process(sig, sampling_rate=FS["RSP"])

    rate = rsp["RSP_Rate"]
    amp = rsp["RSP_Amplitude"]

    out["RSP_Rate_Mean"] = rate.mean()
    out["RSP_Rate_Std"] = rate.std()
    out["RSP_Rate_CV"] = rate.std() / rate.mean() if rate.mean() else np.nan

    out["RSP_Amplitude_Mean"] = amp.mean()
    out["RSP_Amplitude_Std"] = amp.std()

    out["RSP_Volume_Proxy"] = (amp * rate / 60.0).mean()
    # RVT : Respiratory Volume per Time
    try:
        rvt = nk.rsp_rvt(sig, sampling_rate=FS["RSP"])
    
        if isinstance(rvt, pd.DataFrame):
            col = "RSP_RVT" if "RSP_RVT" in rvt.columns else rvt.columns[0]
            out["RSP_RVT"] = np.nanmean(rvt[col].to_numpy(float))
    
        elif isinstance(rvt, pd.Series):
            out["RSP_RVT"] = np.nanmean(rvt.to_numpy(float))
    
        else:
            out["RSP_RVT"] = np.nanmean(np.asarray(rvt, dtype=float))
    
    except Exception:
        # fFallback : approximation volumétrique par produit Amplitude * Fréquence`
        out["RSP_RVT"] = np.nanmean(amp * rate / 60.0)


    out["RSP_Skewness"] = skew(sig)
    out["RSP_Kurtosis"] = kurtosis(sig)

    if "RSP_Phase" in rsp.columns:
        ph = rsp["RSP_Phase"].to_numpy()

        di = np.sum(ph == 1) / FS["RSP"]
        de = np.sum(ph == 0) / FS["RSP"]

        out["RSP_Phase_Duration_Inspiration"] = di
        out["RSP_Phase_Duration_Expiration"] = de
        out["RSP_Phase_Duration_Ratio"] = di / de if de else np.nan

    try:
        rrv = nk.rsp_rrv(rsp, sampling_rate=FS["RSP"])

        for src, dst in [
            ("RRV_SD1", "RSP_Poincare_SD1"),
            ("RRV_SD2", "RSP_Poincare_SD2"),
        ]:
            if src in rrv.columns:
                out[dst] = rrv[src].iloc[0]

    except Exception:
        pass

    return out, rsp


## RSA optionnel

In [36]:
def feats_rsa(rpeaks, n_ecg, rsp_signals):
    out = {}

    if rpeaks is None or rsp_signals is None:
        return out

    try:
        r_bin = np.zeros(n_ecg, dtype=int)
        rpeaks = np.asarray(rpeaks, dtype=int)
        r_bin[rpeaks[rpeaks < n_ecg]] = 1

        rsa = nk.hrv_rsa(
            pd.DataFrame({"ECG_R_Peaks": r_bin}),
            rsp_signals,
            sampling_rate=FS["ECG"],
            continuous=False
        )

        for k in ["RSA_P2T_Mean", "RSA_P2T_SD", "RSA_Gates_Mean"]:
            if k in rsa:
                out[k] = rsa[k]

    except Exception:
        pass

    return out


## Stats d'erreurs de fenêtrage

In [37]:
from collections import Counter

stats = Counter()

def inc(config, modalite, raison, n=1):
    stats[(config, modalite, raison)] += int(n)

def stats_to_df(stats):
    return (
        pd.DataFrame(
            [(c, m, r, n) for (c, m, r), n in stats.items()],
            columns=["config", "modalite", "raison", "n"]
        )
        .sort_values(["config", "modalite", "raison"])
    )


def valider_et_nettoyer_fenetre(sub, modalite, win_ms, fs):
    """
    1. Vérifie le % de couverture minimum.
    2. Vérification de la durée max du plus grand trou consécutif (sauts Time_ms et NaNs).
    3. Si valide, interpole les petits trous restants sur toutes les colonnes numériques.
    """
    if sub is None or len(sub) == 0:
        return None, "sub_missing"

    rules = RULES_QUALITE[modalite]
    min_cover = rules["min_cover"]
    max_gap_s = rules["max_gap_s"]

    # 1. Taux de couverture (% minimum)
    n_attendus = (win_ms / 1000.0) * fs
    if (len(sub) / n_attendus) < min_cover:
        return None, "not_enough_samples"

    # 2. Trous de timestamps (Sauts dans Time_ms)
    if "Time_ms" in sub.columns:
        time_diffs_s = sub["Time_ms"].diff().dropna() / 1000.0
        if len(time_diffs_s) > 0 and time_diffs_s.max() > max_gap_s:
            return None, "continuous_gap_too_large"

    # 3. Trous sous forme de NaNs consécutifs
    col_sig = "ECG_clean" if modalite == "ECG" else ("RSP_clean" if modalite == "RSP" else None)
    if modalite == "fNIRS":
        cols_fnirs = [c for c in sub.columns if "O2HB" in c.upper()]
        col_sig = cols_fnirs[0] if cols_fnirs else None

    if col_sig and col_sig in sub.columns:
        is_nan = sub[col_sig].isna().to_numpy()
        if np.any(is_nan):
            max_nan_samples = max([len(list(g)) for k, g in itertools.groupby(is_nan) if k], default=0)
            if (max_nan_samples / fs) > max_gap_s:
                return None, "continuous_gap_too_large"

    # 4. Interpolation linéaire sécurisée des petits trous
    sub_clean = sub.copy()
    num_cols = sub_clean.select_dtypes(include=[np.number]).columns
    sub_clean[num_cols] = sub_clean[num_cols].interpolate(method="linear", limit_direction="both")

    return sub_clean, "ok"


def check_window(config, modalite, df, sub, win_ms, fs):
    if df is None:
        inc(config, modalite, "file_missing")
        return False, None
    if "Time_ms" not in df.columns:
        inc(config, modalite, "time_column_missing")
        return False, None

    sub_clean, raison = valider_et_nettoyer_fenetre(sub, modalite, win_ms, fs)
    
    if raison != "ok":
        inc(config, modalite, raison)
        return False, None

    return True, sub_clean

def log_features(config, modalite, f):
    if f is None or len(f) == 0:
        inc(config, modalite, "no_features_extracted")
    else:
        inc(config, modalite, "ok")

## 9. Création d’un dataset complet pour une config

In [38]:
def construire_epochs_config(config):
    rows = []
    local_stats = Counter()

    global stats
    old_stats = stats
    stats = local_stats

    CONFIG = config["nom"]

    step_ms = config["step_ms"]
    win_ms = config["win_ms"]
    wmax_ms = max(win_ms.values())

    rel_centers = centers_relatifs(
        duree_ms=DUREE_AVANT_MS,
        step_ms=step_ms,
        wmax_ms=wmax_ms,
        edge=EDGE_MODE
    )

    participants = sorted(df_temps["Participant"].dropna().astype(int).unique())

    for num in tqdm(participants, desc=config["nom"]):
        data = dico_data.get(num, {})

        df_n = data.get("fNIRS")
        df_e = data.get("ECG")
        df_r = data.get("RSP")

        has_nirs = df_n is not None
        has_ecg = df_e is not None
        has_rsp = df_r is not None

        if not has_nirs:
            n = len(KSS_ANCHORS) * len(rel_centers)
            for m in ["fNIRS", "ECG", "RSP", "RSA"]:
                inc(CONFIG, m, "participant_skipped_no_fnirs", n)
            continue

        bases = detecter_bases_fnirs(df_n)

        for kss_name in KSS_ANCHORS:
            bloc_kss = bloc_attendu_kss(kss_name)

            t_nirs = get_anchor_time(num, "fNIRS", kss_name) if has_nirs else None
            t_ecg = get_anchor_time(num, "ECG", kss_name) if has_ecg else None
            t_rsp = get_anchor_time(num, "RSP", kss_name) if has_rsp else None

            if t_nirs is None and t_ecg is None and t_rsp is None:
                n = len(rel_centers)
                inc(CONFIG, "fNIRS", "anchor_nan" if has_nirs else "file_missing", n)
                inc(CONFIG, "ECG",   "anchor_nan" if has_ecg else "file_missing", n)
                inc(CONFIG, "RSP",   "anchor_nan" if has_rsp else "file_missing", n)
                inc(CONFIG, "RSA",   "anchor_nan_or_missing_ecg_rsp", n)
                continue

            for ei, c_rel in enumerate(rel_centers):
                row = {
                    "Participant": num,
                    "KSS": kss_name,
                    "Epoch": ei,
                    "Epoch_center_rel_ms": round(c_rel, 2),
                    "Epoch_center_rel_s": round(c_rel / 1000, 2),
                    "has_fNIRS_file": int(has_nirs),
                    "has_ECG_file": int(has_ecg),
                    "has_RSP_file": int(has_rsp),
                    "step_ms": step_ms,
                    "win_fNIRS_ms": win_ms["fNIRS"],
                    "win_ECG_ms": win_ms["ECG"],
                    "win_RSP_ms": win_ms["RSP"],
                }

                rpeaks = None
                rsp_sig = None
                n_ecg = None

                # -------------------------
                # fNIRS
                # -------------------------
                if not has_nirs:
                    inc(CONFIG, "fNIRS", "file_missing")
                elif t_nirs is None:
                    inc(CONFIG, "fNIRS", "anchor_nan")
                else:
                    center_abs = t_nirs + c_rel
                    sub = slice_window(df_n, center_abs, win_ms["fNIRS"], bloc=bloc_kss)
                    mixed = False
                    if sub is not None and "Bloc_norm" in sub.columns:
                        if len(sub["Bloc_norm"].dropna().unique()) > 1:
                            inc(CONFIG, "fNIRS", "mixed_bloc")
                            mixed = True

                    ok_n, sub_n_clean = check_window(CONFIG, "fNIRS", df_n, sub, win_ms["fNIRS"], FS["fNIRS"])
                    if not mixed and ok_n:
                        try:
                            f = feats_fnirs(sub_n_clean, bases)
                            row.update(f)
                            log_features(CONFIG, "fNIRS", f)
                        except Exception as ex:
                            inc(CONFIG, "fNIRS", "feature_error")
                            print(f"fNIRS P{num}/{kss_name}/epoch{ei}: {ex}")

                # -------------------------
                # ECG
                # -------------------------
                if not has_ecg:
                    inc(CONFIG, "ECG", "file_missing")
                elif t_ecg is None:
                    inc(CONFIG, "ECG", "anchor_nan")
                else:
                    center_abs = t_ecg + c_rel
                    sub = slice_window(df_e, center_abs, win_ms["ECG"], bloc=bloc_kss)
                    mixed = False
                    if sub is not None and "Bloc_norm" in sub.columns:
                        if len(sub["Bloc_norm"].dropna().unique()) > 1:
                            inc(CONFIG, "ECG", "mixed_bloc")
                            mixed = True

                    ok_e, sub_e_clean = check_window(CONFIG, "ECG", df_e, sub, win_ms["ECG"], FS["ECG"])
                    if not mixed and ok_e:
                        try:
                            f, rpeaks = feats_ecg(sub_e_clean)
                            row.update(f)
                            n_ecg = len(sub_e_clean)
                            log_features(CONFIG, "ECG", f)
                        except Exception as ex:
                            inc(CONFIG, "ECG", "feature_error")
                            print(f"ECG P{num}/{kss_name}/epoch{ei}: {ex}")

                # -------------------------
                # RSP
                # -------------------------
                if not has_rsp:
                    inc(CONFIG, "RSP", "file_missing")
                elif t_rsp is None:
                    inc(CONFIG, "RSP", "anchor_nan")
                else:
                    center_abs = t_rsp + c_rel
                    sub = slice_window(df_r, center_abs, win_ms["RSP"], bloc=bloc_kss)
                    mixed = False
                    if sub is not None and "Bloc_norm" in sub.columns:
                        if len(sub["Bloc_norm"].dropna().unique()) > 1:
                            inc(CONFIG, "RSP", "mixed_bloc")
                            mixed = True

                    ok_r, sub_r_clean = check_window(CONFIG, "RSP", df_r, sub, win_ms["RSP"], FS["RSP"])
                    if not mixed and ok_r:
                        try:
                            f, rsp_sig = feats_rsp(sub_r_clean)
                            row.update(f)
                            log_features(CONFIG, "RSP", f)
                        except Exception as ex:
                            inc(CONFIG, "RSP", "feature_error")
                            print(f"RSP P{num}/{kss_name}/epoch{ei}: {ex}")

                # -------------------------
                # RSA
                # -------------------------
                if INCLURE_RSA:
                    if rpeaks is not None and rsp_sig is not None and n_ecg is not None:
                        f = feats_rsa(rpeaks, n_ecg, rsp_sig)
                        row.update(f)
                        log_features(CONFIG, "RSA", f)
                    else:
                        inc(CONFIG, "RSA", "missing_ecg_or_rsp")

                rows.append(row)

    df_epochs = pd.DataFrame(rows)

    if not df_epochs.empty:
        df_epochs = df_epochs.merge(kss_long, on=["Participant", "KSS"], how="left")
        df_epochs["Groupe"] = df_epochs["Participant"].map(map_groupe)

    df_stats = stats_to_df(local_stats)
    stats = old_stats

    return df_epochs, df_stats

## 10. Construction des combinaisons de modalités

In [39]:
META_COLS = [
    "Participant",
    "KSS",
    "Epoch",
    "Epoch_center_rel_ms",
    "Epoch_center_rel_s",
    "Groupe",
    "KSS_label",
    "has_fNIRS_file",
    "has_ECG_file",
    "has_RSP_file",
    "step_ms",
    "win_fNIRS_ms",
    "win_ECG_ms",
    "win_RSP_ms",
]

PREF = {
    "fNIRS": ("fNIRS_",),
    "ECG": ("ECG_", "HRV_"),
    "RSP": ("RSP_",),
}

PREF_RSA = ("RSA_",)


def colonnes_modalite(df, modalite):
    return [
        c for c in df.columns
        if any(c.startswith(p) for p in PREF[modalite])
    ]


def build_dataset(df_epochs, modalites):
    cols = []

    for m in modalites:
        cols += colonnes_modalite(df_epochs, m)

    if "ECG" in modalites and "RSP" in modalites:
        cols += [c for c in df_epochs.columns if c.startswith(PREF_RSA)]

    cols = sorted(set(cols))

    meta = [c for c in META_COLS if c in df_epochs.columns]

    out = df_epochs[meta + cols].copy()

    # on garde uniquement les lignes où chaque modalité demandée existe vraiment
    for m in modalites:
        mcols = colonnes_modalite(out, m)

        if len(mcols) == 0:
            return out.iloc[0:0].copy()

        out = out[out[mcols].notna().any(axis=1)].copy()

    return out


## 11. Boucle finale : crée tous les datasets automatiquement

In [40]:
COMBINAISONS = {
    "nirs": ["fNIRS"],
    "nirs_ecg_rsp": ["fNIRS", "ECG", "RSP"],
    "nirs_ecg": ["fNIRS", "ECG"],
    "nirs_rsp": ["fNIRS", "RSP"],
}


In [41]:
for config in CONFIGS_FENETRAGE:
    nom_config = config["nom"]

    print("\n" + "=" * 80)
    print(f"Création config : {nom_config}")
    print("=" * 80)

    dossier_out = DOSSIER_DATASETS / nom_config
    dossier_out.mkdir(parents=True, exist_ok=True)

    df_epochs, df_stats = construire_epochs_config(config)


    # fichier complet avec toutes les features disponibles
    path_all = dossier_out / "df_epochs_all_modalites.csv"
    df_epochs.to_csv(path_all, index=False)
    print(f"✅ complet : {path_all} | shape={df_epochs.shape}")

    # datasets par combinaison
    for nom_ds, modalites in COMBINAISONS.items():
        ds = build_dataset(df_epochs, modalites)

        path_ds = dossier_out / f"ds_{nom_ds}.csv"
        ds.to_csv(path_ds, index=False)

        print(f"✅ {nom_ds} : {path_ds.name} | shape={ds.shape}")
    
    # résumé config
    pd.DataFrame([{
        "nom": nom_config,
        "step_ms": config["step_ms"],
        "win_fNIRS_ms": config["win_ms"]["fNIRS"],
        "win_ECG_ms": config["win_ms"]["ECG"],
        "win_RSP_ms": config["win_ms"]["RSP"],
        "duree_avant_ms": DUREE_AVANT_MS,
        "edge_mode": EDGE_MODE,
        "rules_qualite": str(RULES_QUALITE),  
    }]).to_csv(dossier_out / "config.csv", index=False)

    #résumé stats erreurs :
    df_stats.to_csv(dossier_out / "stats_exclusions.csv", index=False)
    display(df_stats)


print("\n🎉 Tous les datasets sont créés.")



Création config : short_step10_nirs20_bio60


short_step10_nirs20_bio60:  15%|█▌        | 9/60 [07:47<45:58, 54.10s/it]

RSP P10/KSS3/epoch1: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch4: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch5: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch7: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch8: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch11: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch12: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch14: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch15: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch18: index -1 is out of bounds for axis 0 with size 0
RSP P10/KSS3/epoch19: index 1 is out of bounds for axis 0 with size 1
RSP P10/KSS3/epoch20: index 1 is out of bounds for axis 0 with size 1
RSP P10/KSS3/epoch21: index 1 is out of bounds for axis 0 with size 1


short_step10_nirs20_bio60: 100%|██████████| 60/60 [51:40<00:00, 51.68s/it]


✅ complet : DATASETS_cover&interpolation\short_step10_nirs20_bio60\df_epochs_all_modalites.csv | shape=(7200, 857)
✅ nirs : ds_nirs.csv | shape=(7118, 814)
✅ nirs_ecg_rsp : ds_nirs_ecg_rsp.csv | shape=(6099, 857)
✅ nirs_ecg : ds_nirs_ecg.csv | shape=(6120, 840)
✅ nirs_rsp : ds_nirs_rsp.csv | shape=(6219, 828)


...

RSP P9/KSS6/epoch9: Number of samples, -1, must be non-negative.


long_step30_nirs60_bio180: 100%|██████████| 60/60 [43:06<00:00, 43.11s/it]


✅ complet : DATASETS_cover&interpolation\long_step30_nirs60_bio180\df_epochs_all_modalites.csv | shape=(2400, 857)
✅ nirs : ds_nirs.csv | shape=(2364, 814)
✅ nirs_ecg_rsp : ds_nirs_ecg_rsp.csv | shape=(2019, 857)
✅ nirs_ecg : ds_nirs_ecg.csv | shape=(2022, 840)
✅ nirs_rsp : ds_nirs_rsp.csv | shape=(2059, 828)


,config,modalite,raison,n
1,long_step30_nirs60_bio180,ECG,file_missing,320
7,long_step30_nirs60_bio180,ECG,not_enough_samples,16
4,long_step30_nirs60_bio180,ECG,ok,2044
11,long_step30_nirs60_bio180,ECG,sub_missing,20
3,long_step30_nirs60_bio180,RSA,missing_ecg_or_rsp,359
6,long_step30_nirs60_bio180,RSA,ok,2041
10,long_step30_nirs60_bio180,RSP,feature_error,1
2,long_step30_nirs60_bio180,RSP,file_missing,280
8,long_step30_nirs60_bio180,RSP,not_enough_samples,18
5,long_step30_nirs60_bio180,RSP,ok,2081



🎉 Tous les datasets sont créés.


In [42]:
df_stats.pivot_table(
    index=["config", "modalite"],
    columns="raison",
    values="n",
    aggfunc="sum",
    fill_value=0
)


raison                              continuous_gap_too_large  feature_error  \
config                    modalite                                            
long_step30_nirs60_bio180 ECG                              0              0   
                          RSA                              0              0   
                          RSP                              0              1   
                          fNIRS                            9              0   

raison                              file_missing  missing_ecg_or_rsp  \
config                    modalite                                     
long_step30_nirs60_bio180 ECG                320                   0   
                          RSA                  0                 359   
                          RSP                280                   0   
                          fNIRS                0                   0   

raison                              not_enough_samples    ok  sub_missing  
config                    modalite                                         
long_step30_nirs60_bio180 ECG                       16  2044           20  
                          RSA                        0  2041            0  
                          RSP                       18  2081           20  
                          fNIRS                      7  2364           20

# CREATION DATASET DE CALIBRATION BASELINE + Début PVT

In [43]:
# ============================================================
# PARAMÈTRES CALIBRATION
# ============================================================

CALIB_PRE_MS = 60_000       # 1 min baseline avant PVT
CALIB_PVT_MS = 300_000      # 5 min début PVT
CALIB_START_MS = -CALIB_PRE_MS
CALIB_END_MS = CALIB_PVT_MS
EDGE_CALIB = "inside"

# À ADAPTER aux noms exacts dans ton fichier temps
CALIB_ANCHORS = {
    "pre": "Pré - Début P1",
    "post": "Post - Début P1",
}

def centers_interval(t_start_ms, t_end_ms, step_ms, win_ms, edge="inside"):
    """
    Centres de fenêtres dans un intervalle relatif [t_start_ms ; t_end_ms].
    edge='inside' => toute la fenêtre doit être contenue dans l'intervalle.
    """
    if edge == "inside":
        lo = t_start_ms + win_ms / 2
        hi = t_end_ms - win_ms / 2
    else:
        lo = t_start_ms + step_ms / 2
        hi = t_end_ms

    if hi < lo:
        return np.array([])

    return np.arange(lo, hi + 1e-9, step_ms)


def get_calib_anchor_time(num, modality, bloc_calib):
    """
    Récupère le temps absolu du début PVT pour pre ou post.
    """
    anchor_col = CALIB_ANCHORS[bloc_calib]
    kss_ref = "KSS1" if bloc_calib == "pre" else "KSS6"

    row_time = get_row_temps(num, modality, kss_ref)

    if row_time is None:
        return None

    if anchor_col not in row_time.index:
        return None

    if pd.isna(row_time[anchor_col]):
        return None

    return float(row_time[anchor_col])


In [44]:
def construire_calibration_config(config):
    rows = []

    CONFIG = config["nom"]
    step_ms = config["step_ms"]
    win_ms = config["win_ms"]

    participants = sorted(df_temps["Participant"].dropna().astype(int).unique())

    for num in tqdm(participants, desc=f"calibration_{CONFIG}"):

        data = dico_data.get(num, {})

        df_n = data.get("fNIRS")
        df_e = data.get("ECG")
        df_r = data.get("RSP")

        has_nirs = df_n is not None and len(df_n) > 0
        has_ecg = df_e is not None and len(df_e) > 0
        has_rsp = df_r is not None and len(df_r) > 0

        bases = detecter_bases_fnirs(df_n) if has_nirs else []

        for bloc_calib in ["pre", "post"]:

            bloc_norm = bloc_calib

            t_nirs = get_calib_anchor_time(num, "fNIRS", bloc_calib)
            t_ecg = get_calib_anchor_time(num, "ECG", bloc_calib)
            t_rsp = get_calib_anchor_time(num, "RSP", bloc_calib)

            # 1) fNIRS
            if has_nirs and t_nirs is not None:
                centers_nirs = centers_interval(
                    CALIB_START_MS, CALIB_END_MS, step_ms, win_ms["fNIRS"], edge=EDGE_CALIB
                )

                for ei, c_rel in enumerate(centers_nirs):
                    c_abs = t_nirs + c_rel
                    sub_n = slice_window(df_n, c_abs, win_ms["fNIRS"], bloc=bloc_norm)

                    ok_n, sub_n_clean = check_window(CONFIG, "fNIRS", df_n, sub_n, win_ms["fNIRS"], FS["fNIRS"])
                    if not ok_n:
                        continue

                    row = {
                        "Participant": num,
                        "Calibration_bloc": bloc_calib,
                        "Calibration_modality": "fNIRS",
                        "Calibration_epoch": ei,
                        "Calibration_center_rel_ms": c_rel,
                        "Calibration_center_rel_s": c_rel / 1000,
                        "Calibration_center_abs_ms": c_abs,
                        "Config": CONFIG,
                        "step_ms": step_ms,
                        "win_fNIRS_ms": win_ms["fNIRS"],
                        "win_ECG_ms": win_ms["ECG"],
                        "win_RSP_ms": win_ms["RSP"],
                    }

                    row.update(feats_fnirs(sub_n_clean, bases))
                    rows.append(row)

            # 2) BIO (ECG + RSP + RSA)
            has_bio_anchor = t_ecg is not None or t_rsp is not None

            if has_bio_anchor and (has_ecg or has_rsp):
                win_bio_ms = max(win_ms["ECG"], win_ms["RSP"])
                centers_bio = centers_interval(
                    CALIB_START_MS, CALIB_END_MS, step_ms, win_bio_ms, edge=EDGE_CALIB
                )

                for ei, c_rel in enumerate(centers_bio):
                    row = {
                        "Participant": num,
                        "Calibration_bloc": bloc_calib,
                        "Calibration_modality": "BIO",
                        "Calibration_epoch": ei,
                        "Calibration_center_rel_ms": round(c_rel, 2),
                        "Calibration_center_rel_s": round(c_rel / 1000, 2),
                        "Config": CONFIG,
                        "step_ms": step_ms,
                        "win_fNIRS_ms": win_ms["fNIRS"],
                        "win_ECG_ms": win_ms["ECG"],
                        "win_RSP_ms": win_ms["RSP"],
                    }

                    valid_any = False
                    rpeaks = None
                    rsp_sig = None
                    n_ecg = None

                    # ECG
                    if has_ecg and t_ecg is not None:
                        center_abs = t_ecg + c_rel
                        sub = slice_window(df_e, center_abs, win_ms["ECG"], bloc=bloc_norm)
                        mixed = False
                        if sub is not None and "Bloc_norm" in sub.columns:
                            if len(sub["Bloc_norm"].dropna().unique()) > 1:
                                mixed = True

                        ok_e, sub_e_clean = check_window(CONFIG, "ECG", df_e, sub, win_ms["ECG"], FS["ECG"])
                        if not mixed and ok_e:
                            try:
                                f, rpeaks = feats_ecg(sub_e_clean)
                                row.update(f)
                                n_ecg = len(sub_e_clean)
                                valid_any = True
                            except Exception as ex:
                                inc(CONFIG, "ECG", "feature_error")
                                print(f"ECG calibration P{num}/{bloc_calib}/epoch{ei}: {ex}")

                    # RSP
                    if has_rsp and t_rsp is not None:
                        center_abs = t_rsp + c_rel
                        sub = slice_window(df_r, center_abs, win_ms["RSP"], bloc=bloc_norm)
                        mixed = False
                        if sub is not None and "Bloc_norm" in sub.columns:
                            if len(sub["Bloc_norm"].dropna().unique()) > 1:
                                mixed = True

                        ok_r, sub_r_clean = check_window(CONFIG, "RSP", df_r, sub, win_ms["RSP"], FS["RSP"])
                        if not mixed and ok_r:
                            try:
                                f, rsp_sig = feats_rsp(sub_r_clean)
                                row.update(f)
                                valid_any = True
                            except Exception as ex:
                                inc(CONFIG, "RSP", "feature_error")
                                print(f"RSP calibration P{num}/{bloc_calib}/epoch{ei}: {ex}")

                    # RSA
                    if INCLURE_RSA:
                        if rpeaks is not None and rsp_sig is not None and n_ecg is not None:
                            try:
                                f = feats_rsa(rpeaks, n_ecg, rsp_sig)
                                row.update(f)
                                valid_any = True
                            except Exception as ex:
                                inc(CONFIG, "RSA", "feature_error")
                                print(f"RSA calibration P{num}/{bloc_calib}/epoch{ei}: {ex}")

                    if valid_any:
                        rows.append(row)

    df_calib = pd.DataFrame(rows)
    return df_calib

In [45]:
for config in CONFIGS_FENETRAGE:

    nom_config = config["nom"]
    dossier_out = DOSSIER_DATASETS / nom_config
    dossier_out.mkdir(parents=True, exist_ok=True)

    df_calib = construire_calibration_config(config)

    path_calib = dossier_out / "calibration_epochs_all_modalites.csv"
    df_calib.to_csv(path_calib, index=False)

    print(f"{nom_config} | calibration saved: {path_calib} | shape={df_calib.shape}")

    display(
        df_calib
        .groupby(["Calibration_bloc", "Calibration_modality"])
        .size()
        .reset_index(name="n_windows")
    )


calibration_short_step10_nirs20_bio60:  13%|█▎        | 8/60 [03:58<28:25, 32.79s/it]

RSP calibration P9/pre/epoch0: index -1 is out of bounds for axis 0 with size 0
RSP calibration P9/pre/epoch1: index -1 is out of bounds for axis 0 with size 0
RSP calibration P9/pre/epoch4: index -1 is out of bounds for axis 0 with size 0
RSP calibration P9/pre/epoch5: index -1 is out of bounds for axis 0 with size 0
RSP calibration P9/pre/epoch8: index -1 is out of bounds for axis 0 with size 0
RSP calibration P9/pre/epoch11: index -1 is out of bounds for axis 0 with size 0
RSP calibration P9/pre/epoch12: index 1 is out of bounds for axis 0 with size 1
RSP calibration P9/pre/epoch13: index 1 is out of bounds for axis 0 with size 1
RSP calibration P9/pre/epoch14: index 1 is out of bounds for axis 0 with size 1


calibration_short_step10_nirs20_bio60:  15%|█▌        | 9/60 [04:28<27:03, 31.84s/it]

RSP calibration P10/pre/epoch0: index -1 is out of bounds for axis 0 with size 0
RSP calibration P10/pre/epoch1: index -1 is out of bounds for axis 0 with size 0
RSP calibration P10/pre/epoch3: index -1 is out of bounds for axis 0 with size 0


calibration_short_step10_nirs20_bio60: 100%|██████████| 60/60 [28:56<00:00, 28.93s/it]


...

medium_step15_nirs30_bio120 | calibration saved: DATASETS_cover&interpolation\medium_step15_nirs30_bio120\calibration_epochs_all_modalites.csv | shape=(4491, 855)


,Calibration_bloc,Calibration_modality,n_windows
0,post,BIO,884
1,post,fNIRS,1327
2,pre,BIO,901
3,pre,fNIRS,1379


calibration_long_step30_nirs60_bio180: 100%|██████████| 60/60 [18:46<00:00, 18.78s/it]


long_step30_nirs60_bio180 | calibration saved: DATASETS_cover&interpolation\long_step30_nirs60_bio180\calibration_epochs_all_modalites.csv | shape=(2029, 855)


,Calibration_bloc,Calibration_modality,n_windows
0,post,BIO,364
1,post,fNIRS,634
2,pre,BIO,371
3,pre,fNIRS,660
